<font color ='Navy'><center> __BruFace__ </center></font>
<br> MA1 Mechanical Engineering <br>  __Mechanical vibrations__ <br>
***
***
<font color='Navy'><h1><center>Session 4 : Modal space </center></h1></font> 
***
<div class="alert alert-block alert-info">  
<center><b>Welcome to the fourth session. The primary focus of today's session is to delve into modal space.</b></center>  
</div>  

# <u>Table of contents</u>
1. [Recap](#recap)
2. [Modal space projection](#modal)
    1. [projection](#projection)
    1. [time domain response](#time)
    2. [frequency response](#freq)

In [ ]:
import sys
import os

# Only if the session is opened in google colab link
if "google.colab" in sys.modules:
    repo = "/content/MV2627_Exercises"
    if not os.path.exists(repo):
        !git clone -q https://github.com/OWI-Lab/MV2627_Exercises.git {repo}
    %cd /content/MV2627_Exercises/Session4_MDOF2
    !pip install -q uv
    !uv pip install --system ipympl ipywidgets jupyterlab-widgets
    from google.colab import output
    output.enable_custom_widget_manager()

In [ ]:
# let's import both numpy, matplotlib.pyplot and scipy's linalg
# the "as" in the code bellow is to tell python that if we reference to the package using np, plt and linalg
import numpy as np
import matplotlib.pyplot as plt 
import scipy.linalg as linalg

# **Recap**<a class="anchor" id="modal"></a>
## Understanding Mode Shapes with Animation.

To better understand the physical meaning of **mode shapes** we will return to the 3DOF system introduced in the previous session.

<img src="./Images/MDOF.png" width=200 height=200 />

Now let us have a look at  animations that visualize how each mode behaves. For Mode 1, note how the three masses are moving in-phase with respect to each other.


In [ ]:
from IPython.display import display, HTML
import os 
folder = r"Animation"
gif_files = ["mode1.gif"]
html = "<div style='display: flex; justify-content: center;'>"
for gif in gif_files:
    path = os.path.join(folder,gif)
    html += f"<div style='margin: `px;'><img src='{path}' style='width:1000px;'></div>"
html += "</div>"
# Display the HTML
display(HTML(html))

For Mode 2, mass 1 and 2 are in phase but mass 3 is out of phase. 


In [ ]:
gif_files = ["mode2.gif"]
html = "<div style='display: flex; justify-content: center;'>"
for gif in gif_files:
    path = os.path.join(folder,gif)
    html += f"<div style='margin: `px;'><img src='{path}' style='width:1000px;'></div>"
html += "</div>"
# Display the HTML
display(HTML(html))

Similarily for Mode 3, mass 2 is out of phase with mass 1 and 3.


In [ ]:
gif_files = ["mode3.gif"]
html = "<div style='display: flex; justify-content: center;'>"
for gif in gif_files:
    path = os.path.join(folder,gif)
    html += f"<div style='margin: `px;'><img src='{path}' style='width:1000px;'></div>"
html += "</div>"
# Display the HTML
display(HTML(html))

---

Now that we have **visualized** and **interpreted** the three mode shapes of our 3-DOF system, we are ready to put that insight to work. In this session, we will:

1. **Decouple** the full MDOF equations by projecting mass, damping and stiffness onto the modal basis.  
2. **Compute** the impulse responses of the resulting uncoupled SDOF systems.  
3. **Reconstruct** both time- and frequency-domain responses back in the physical (nodal) space.  
4. **Apply** an arbitrary excitation (a swept-sine “chirp” on DOF 3) to demonstrate how modal superposition predicts the system’s behavior.

By the end of this session, you will see how modal projection turns a complex coupled problem into simple SDOF analyses—and how to reassemble the pieces to recover the full MDOF response.

---

### Modal Projection <a class="anchor" id="modal"></a>

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

<b>Assignment 1.1:</b> Projecting the MDOF system onto the modal space <br>

In this exercice we will use the real mode shapes (obtained from solving the conservative eigenvalue problem) to project the equation of motion in the modal basis. <br>

Complete the function `project_matrix` below to project a given mass, stiffness or damping matrix into the modal space using the matrix of eigenvectors `psi`: <br>
    
**TIP**: Use the `@` operator to perform matrix multiplication and .T to transpose a matrix.

```python

array1 @ array2 == np.matmul(array1, array2) 
array.T == np.transpose(array)
```

In [ ]:
def project_matrix(A:np.ndarray,psi:np.ndarray):
    """ A function that project a matrix A onto the modal space defined by psi
    Arguments:
    A -- The matrix to be projected into the modal space, can be M, C or K
    psi -- matrix of real modal shapes
    
    Returns:
    A_prj -- The projected matrix onto the modal space
    """

    # YOUR CODE HERE
    raise NotImplementedError()

    return A_prj

Let us check that you implemented `project_matrix` correctly.

In [ ]:
A_test = np.array([[1,2,3], [0, 1, 0], [0, 0, -1]])  
psi_test = np.array([[1, 0, 1], [0, np.sqrt(2), 0], [1, 0, -1]])  / np.sqrt(2)
A_prj_test = project_matrix(A_test,psi_test)
assert np.allclose(np.trace(A_prj_test), np.trace(A_test),rtol=1e-3), 'project_matrix is not implemented correctly' # (the divergence of the two matrices should be the same) FYI trace = divergence of a matrix (assuming psi.T  = inv(psi) )
assert np.allclose(np.linalg.norm(A_test, 'fro'), np.linalg.norm(A_prj_test, 'fro')), "Frobenius norm should be preserved for orthogonal projections." # RMS of eigenvalues should be preserved

Before we can project into modal space, let us set up our MDOF system. In session 3 we have already written functions to assemble the mass, damping and stiffness matrices, solve for natural frequencies and mode shapes, and convert eigenvalues into resonance frequencies. 

Let us import them from the session3:

In [ ]:
# Import the previously created functions here
from session3 import build_3DOF_matrices, calc_eigenvalues_eigenvectors, lambda_to_resonance_frequency
# lambda_to_resonance_frequency is a new function that I added for free :) 

A quick usage example: 

```python
M, C, K = build_3DOF_matrices(m=10,c=10,c1=10,k=10,k1=10) # compute the mass, damping and stiffness matrices
lamb, psi = calc_eigenvalues_eigenvectors(M,K) # compute the eigenvalues and eigenvectors
freq_res = lambda_to_resonance_frequency(lamb) # compute the resonance frequency
```
The used arguments are just an example, you will need to change them.

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

<b> Assignment 1.2:</b> Let us test your implementation of `project_matrix` by again taking the aforementioned 3DOF system with $m = 1 kg$, $k = k1 = 16 N/m$, $c = c1 = 0.1 N s/m$  and calculating the eigenvalues (`lamb`) and eigenvectors (`psi`) for the conservative system. First build the M, C and K matrices. Next use `calc_eigenvalues_eigenvectors` and `project_matrix` to project the M, C and K matrices onto the modal space. Call these projections `M_prj`, `C_prj` and `K_prj` respectively.

In [ ]:
M , C, K = None, None, None
lamb,psi = None, None
f_res = None
M_prj, K_prj, C_prj = None, None, None

# YOUR CODE HERE
raise NotImplementedError()

The following cell checks your implementation.

In [ ]:
freq_modal = np.sqrt(np.diag(K_prj)/np.diag(M_prj))/(2*np.pi) # computing the resonance frequencies from the projected masses and stiffnesses
assert np.allclose(f_res, freq_modal)

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

<b>Assignment 1.3:</b> What do you expect `M_prj`, `K_prj`, and `C_prj` to look like? Complete the cell below. 

**Tip:** Use `np.diag(np.diag(A))` to extract only the diagonal part of a matrix and convert it back into a diagonal matrix — zeroing out the off-diagonal terms.

The following diagram explains why we need to apply np.diag twice and what does `np.diag` do:

$$
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix}
\quad \xrightarrow{\text{np.diag}} \quad
\begin{bmatrix}
1 & 5 & 9
\end{bmatrix}
\quad \xrightarrow{\text{np.diag}} \quad
\begin{bmatrix}
1 & 0 & 0 \\
0 & 5 & 0 \\
0 & 0 & 9
\end{bmatrix}
$$


In [ ]:
M_prj_is_diagonal = np.allclose(M_prj, np.diag(np.diag(M_prj)))
K_prj_is_diagonal = None 
C_prj_is_diagonal = None 
# YOUR CODE HERE
raise NotImplementedError()
print(f"Is M a diagonal matrix? {M_prj_is_diagonal}")
print(f"Is K diagonal matrix? {K_prj_is_diagonal}")
print(f"Is C diagonal matrix? {C_prj_is_diagonal}")

> <b>Note:</b> All three matrices $M$, $C$ and $K$ can be projected succesfully on the modal space. This is no surprise for $M$ and $K$. But why can we succesfully project $C$ onto the modal space? What will happen when we change the value of $c_1$? 

<details>
<summary><b>Click to get the answer</b></summary>

With $c_1=c$, our $\mathbf{C}$ is proportional to $\mathbf{K}$. This is known as viscous damping, a special case of Rayleigh damping.

Because $\psi\mathbf{K}\psi^T$ is a diagonal matrix, so will $\psi\mathbf{C}\psi^T$.

When we change the value of $c_1$, then $\mathbf{C}$ is no longer proportional to $\mathbf{K}$. As a result, $\psi\mathbf{C}\psi^T$ will no longer be a diagonal matrix, meaning that the modal equations remain coupled and cannot be treated as independent single-degree-of-freedom systems.

</details>


<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.4:**  In the course we have talked about Rayleigh damping. where one can decompose the damping matrix into a linear combination of the mass and stiffness matrix. 
${\bf C}= \alpha {\bf K} + \beta {\bf M}$.
What should the values of $\alpha$ and $\beta$ be to obtain the same damping matrix as in the previous assignment? Fill in the values of $\alpha$ and $\beta$ below in `alpha` and `beta`:

In [ ]:
alpha = None
beta = None
# YOUR CODE HERE
raise NotImplementedError()

And let us check your results (Hidden tests)

In [ ]:
# cell metadata=autograder-tests
assert alpha is not None, 'alpha is not defined'
assert beta is not None, 'beta is not defined'

### Impulse Response in the Modal Space 
Now that we have projected the MDOF system into the modal space, we can use the decoupled SDOF equations to calculate the impulse response of **3 uncoupled SDOF systems/Mode** .

Importing and reusing the function `impulse_response` function you have implemented in session 2, we can calculate the impulse response of any SDOF system and this also works in the modal space. 

Reminder: the impulse response of a SDOF system is given by: 
$$
h(t) = \frac{1}{\omega_n} e^{-\zeta \omega_n t} sin(\omega_d t)$$
where $\omega_d = \omega_n \sqrt{1-\zeta^2}$
where $\omega_d$ is the damped frequency and $\zeta$ is the damping ratio.  
$\zeta = \frac{c}{2 \sqrt{mk}}$ 
where $c$ is the damping coefficient, $m$ is the mass and $k$ is the stiffness.

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

<b>Assignment 1.5:</b> Use the function `impulse_response` to calculate the impulse response of the system in the modal space.
The decoupled impulse‐response in modal space is simply the vector of the three SDOF responses, i.e.

$$
h_{modal}(t) = \begin{bmatrix} 
h_1(t) \\ h_2(t) \\ h_3(t) \end{bmatrix}
$$
where $h_1(t)$, $h_2(t)$ and $h_3(t)$ are the impulse responses of the three SDOF systems, respectively linked to the first, second and third mode of the original 3DOF system.
    
Save the impulse responses in a variable called `h_modal` and plot the impulse response of the system in the modal space.




In [ ]:
from session2 import impulse_response

f_s = 10 # Defining a sampling frequency
t = np.linspace(0,600, f_s*600) # Defining a time vector of 600s, sampled at f_s Hz

h_modal = [] # to add new elements to this list you can use impulse_responses.append
# YOUR CODE HERE
raise NotImplementedError()
h_modal = np.array(h_modal) 

Doing some checks and then plotting the 3 impulse responses.

In [ ]:
# Doing some checks first
assert len(h_modal) ==3, "impulse_responses does not contain the three SDOF reponses"
assert not np.array_equal(h_modal[0], h_modal[1]), "The impulse responses you calculated are not different"
for h_i in h_modal:
    assert len(h_i) == len(t), "The length of your calculated modal response does not match the timevector t"
assert np.isclose(np.sum(np.square(h_modal)), 81.2359), "The impulse response of the first SDOF system is not zero at t=0"


# Plotting the impulse responses of the three SDOF systems and the whole MDOF system
fig, ax = plt.subplots(1, 1, figsize=(8,5))
for i, h_i in enumerate(h_modal):
    ax.plot(t, h_i, label=f'Mode {i+1} ({np.abs(f_res[i]):.2f} Hz)')
ax.legend()
ax.set_xlim([0,60])
ax.set_xlabel('Time (s)')
ax.set_ylabel('h(t)')
ax.grid(which='both', linestyle=':')
_ = ax.set_title('Impulse responses of the 3 SDOF systems')

**Question to check your plot:**  
Examine the impulse responses of the 3 SDOF systems shown in the plot above.
- Do the responses decay over time as expected from a damped system?
- Which mode is dominant in amplitude? Can you explain why?

### Impulse Response in the Nodal Space

Now that we have projected the MDOF system into the modal space and we computed the impulse response of the system in the modal space, we can also project the impulse response back to the nodal space. 

**Nodal space is just another term to describe the physical space.**

We define the impulse response in the nodal space $ h_{i,j}(t) $ as the **impulse response of the original 3DOF system observed from node $ i $** due to an **impulse applied at node $ j $**. Using the modal decomposition of the system, this can be written as:

$$
h_{i,j}(t) = \sum_{r=1}^{3} \psi_{i,r} \, \psi_{j,r} \, h^{\text{modal}}_r(t)
$$

Where:
- $ \psi_{i,r} $: contribution of mode $ r $ to node $ i $  
- $ \psi_{j,r} $: contribution of mode $ r $ to the input node $ j $  
- $ h^{\text{modal}}_r(t) $: impulse response of the $ r^{\text{th}} $ SDOF modal system

This formula shows that each node’s response is a **weighted combination of modal impulse responses $ h^{\text{modal}}_r(t) $**, where the weights are determined by the **mode shapes**. Also note that a 3DOF system has in fact 9 impulse responses, similar to the dimensions of Frequency Response Function (FRF) which we saw in the previous session.


<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

<b>Assignment 1.6:</b> Using the `h_modal` variable from the previous assignment, project the impulse response back to the nodal space using the mode shapes `psi`.
save the impulse response in a variable called `h_nodal`



**TIP:**  
To compute the full impulse response matrix $H(t)$, you need to evaluate $h_{i,j}(t)$ for all combinations of output DOFs $i$ and input DOFs $j$.

This can be done using three nested loops:
- Loop over output DOFs \( i \)
- Loop over input DOFs \( j \)
- Loop over modes \( r \)

At each step, compute the **weighting factor** using the mode shapes: **weight** = $\psi_{i,r} \cdot \psi_{j,r}$

Then update the nodal impulse response using:
```python
h_nodal[:, i, j] += h_modal_r * weight
```
where `h_modal_r` is the impulse response of the $ r^{\text{th}} $ SDOF system.


In [ ]:
f_s = 10 # Defining a sampling frequency
t = np.linspace(0,600, f_s*600) # Defining a time vector of 600s, sampled at f_s Hz

# this list will contain the impulse response of each SDof system
h_nodal = np.zeros((len(t), 3, 3)) # this will be the impulse response of the 3DOF system 
# YOUR CODE HERE
raise NotImplementedError()
h_modal = np.array(h_modal)

> Okay, now we can plot the impulse response in the nodal space and see if it looks like we expect it to look like. We can also compute the FFT of the time domain impulse response and plot the frequency response of the system. 

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(10,5))
plt.tight_layout()
fig2, ax2 = plt.subplots(3, 3, figsize=(10,5)) 
plt.tight_layout()

for i in range(3):
    for j in range(3):
        ax[i,j].plot(t, h_nodal[:,i,j])
        ax[i,j].set_xlim([0,60])
        ax[i,j].set_xlabel('Time (s)')
        ax[i,j].grid(which='both', linestyle=':') 
        freq = np.fft.fftfreq(len(h_nodal[:,i,j]), 1/f_s)
        H_nodal_fft = np.fft.fft(h_nodal[:,i,j]) / len(h_nodal[:,i,j])
        freq_pos = freq[freq >= 0]
        H_nodal_fft = H_nodal_fft[freq >= 0]
        ax2[i,j].plot(freq_pos, np.abs(H_nodal_fft))        
        ax2[i,j].set_xlabel('Frequency (Hz)')
        ax2[i,j].set_ylabel('Magnitude')
        ax2[i,j].grid(which='both', linestyle=':')
        ax2[i,j].set_yscale('log')
        ax2[i,j].set_xlim([0,5])

> **Note:**  
> We can visualize the system's behavior in the time domain by plotting the **nodal impulse response matrix** $ \mathbf{H}(t) $ stored in `h_nodal`.  
> This matrix captures how each degree of freedom (DOF) responds to an impulse applied at every other DOF.
>
> The matrix $\mathbf{H}(t) $ is **symmetric**, meaning:
> Importantly, this impulse response matrix contains the **same information** as the **Frequency Response Function (FRF) matrix**, but in the **time domain**.

**Plot Check**  
Take a moment to reflect on the results you obtained
- Does the **Frequency Response Function (FRF) matrix** look as expected? 
- Does it resemble the FRF behavior you observed in the previous session?
- Does the **time-domain response** match your expectations based on the system's resonance frequencies and damping?
Remember:  
The two sets of plots — time-domain response and frequency-domain FRF — contain the **same information**, just represented in **different domains**.

### Let’s see what we’ve accomplished so far:

- We projected the original MDOF system into the **modal space**, simplifying the dynamics into uncoupled SDOF systems.  
- We computed the **impulse responses** of these 3 SDOF systems.  
- We then used **modal superposition** to reconstruct the full impulse response in the **physical (nodal) space**.


With the two first points, we can now simulate how the 3DOF system responds to **any arbitrary input force**.

---
In order to compute the system's response to an arbitrary input force, we can use the following steps:

   - **Step 1:** Project the input force vector into the modal space using $ \boldsymbol{\psi} $
   - **Step 2:** For each mode, convolve its impulse response $ h^{\text{modal}}_i(t) $ with the corresponding modal force
   - **Step 3:** Reconstruct the physical response via:

     $$
     \mathbf{x}(t) = \boldsymbol{\psi} \, \mathbf{z}(t)
     $$
---
---

In what follows, we’ll implement the above mentiooned steps  to compute the system’s response to a **chirp input** — a swept sine wave applied at the **third DOF, $ x_3 $**.

To generate this signal, we'll use the `chirp` function from `scipy.signal`, creating a frequency-sweeping excitation that starts from 0 Hz and ramps up to 2 Hz over time — perfect for exciting multiple modes across a broad frequency range.
<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.7:** Use chirp to generate a swept sine signal $f_{chirp}(t)$ that <mark style="background-color: #ffff99;"> ranges from 0 to 2 _Hz_, amplitude equal to 1, have it start $t=0$ at a value of $0$; $f_{chirp}(0)=0$,  and make it decreasing at time 0, i.e. $\frac{df_{chirp}}{dt}(0)<0$ </mark>. Reuse the timevector `t` that was introduced in the previous cells. Name the signal `f_chirp`.

**TIP**: You can read the documentation of `chirp` using `?chirp` or find it online.


In [ ]:
from scipy.signal import chirp
f_chirp = None
# YOUR CODE HERE
raise NotImplementedError()
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(t, f_chirp)
ax.set_xlabel('Time (s)')
ax.set_ylabel('$f_{chirp}(t)$')
ax.grid(which='both', linestyle=':')
_ = ax.set_title('Chirp signal')

Let us check your swept sine signal.

In [ ]:
# test chirp
assert len(f_chirp) == len(t),  'The length of the chirp signal does not match the length of the time vector'
assert np.isclose(f_chirp[0], 0), 'Your swept sine does not start at zero! Consider changing the phase'
assert max(f_chirp) <= 1, 'The amplitude of your swept sine is too large'
assert min(f_chirp) >= -1, 'The amplitude of your swept sine is too large'
assert np.isclose(np.abs((np.abs(np.fft.fft(f_chirp)[0:3]).sum())) ,181.847542930501,rtol=1e-10), ' you implemented the chirp signal incorrectly'
assert np.diff(f_chirp)[0] < 0, 'The derivative of the chirp signal in the instant t = 0 should be negative'

Let us now apply this force to the third DOF $x_3$. To do so we must construct the force vector in the physical space. 
<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.8:** Write the function `build_force_vector` that constructs the force vector in physical coordinates. This function assumes only one input location of the force. 
The function takes as input the force applied and the location of the force and returns the force vector in physical coordinates.
</div>

In [ ]:
def build_force_vector(force_time_domain:np.ndarray,n_dof:int, force_location:int):
    """ A function that builds the force vector for a given time domain force vector
    Arguments:
    force_time_domain -- The force vector in the time domain
    n_dof -- The number of degrees of freedom
    force_location -- The location of the force"""
    f = np.zeros((n_dof,len(force_time_domain)))
    # YOUR CODE HERE
    raise NotImplementedError()
    return f

In [ ]:
force_time_domain = np.arange(0,600,1)
assert build_force_vector(force_time_domain,3,1).shape == (3,600), 'The shape of the force vector is not correct'
assert np.allclose(build_force_vector(force_time_domain,3,1)[1,:], force_time_domain), 'The force vector is not built correctly'
assert np.allclose(build_force_vector(force_time_domain,3,1)[0,:], np.zeros(600)), 'The force vector is not built correctly'
assert np.allclose(build_force_vector(force_time_domain,3,1)[2,:], np.zeros(600)), 'The force vector is not built correctly'

Now we need to project this force vector into the modal space.

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.9:** Finish `project_force` to calculate the modal forces for the chirp signal.

</div> 

In [ ]:
def project_force(psi:np.ndarray, force_vector:np.ndarray):
    """
    A function that calculates the modal excitation force vector given the force vector in the physical space
    Arguments:
    psi -- matrix of real modal shapes
    force_vector -- the force vector in the physical space
    """
    # YOUR CODE HERE
    raise NotImplementedError()

Let us check your function; by using some simple checks for a second order system.


In [ ]:
psi_2dof = np.array([[1,1],[1,-1]]) # Textbook mode shapes of a 2DOF system
f_test_1 = np.array([1,2]) # Force vector in the physical space
f_test_2 = np.array([[1]*10,[1]*10]) # Force vector in the physical space
assert np.allclose(project_force(psi_2dof, f_test_1), np.array([3,-1])) # The modal excitation force vector should be [3,1]
assert np.allclose(project_force(psi_2dof, f_test_2), np.array([[2]*10,[0]*10])) # The modal excitation force vector should be [10,0]

Now that we can calculate the modal excitations `project_force(t)` for each of the three modes. Remember that:

$\vec{x}(t) = \sum_{i=1}^n z_i(t)\vec{\psi}_i = \boldsymbol{\psi} \vec{z}(t) $

in which $z_i(t)$ is the SDOF response to the modal force:

$z_i(t) = h_i(t) * f_{modal,i}(t)$


<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.10:** 
Build a function `response_of_set_SDOF` that takes as input the a list of impulse responses `list_of_impulse_responses` and the modal excitation `f_modal` and returns the modal response. 
make sure that this function returns an array and not a list.
</div>

**TIP** 
To convolve two signals you can use `np.convolve(signal1,signal2,model='full')[:len(signal1)]` <br>
To iterate through `impulse_responses` and `force_modal` simultanously, use for the `zip` function

``` python 
for h_i, f_i in zip(impulse_responses, force_modal)
```

In [ ]:
def response_of_many_SDOF(impulse_responses:np.ndarray,force_modal:np.ndarray):
    """
    Calculate the response of each mode by convolving the impulse response with the modal excitation.
    
    Parameters:
    list_of_impulse_responses (list of np.array): List containing the impulse responses of each mode.
    force_modal (np.array): List containing the modal excitations for each mode.
    
    Returns:
    list of np.array: List containing the response of each mode.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

Let's put everything together and plot the response of the each DOF to the swept sine signal applied to the third DOF. 
<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.12:**  
- First compute the force vector in the physical space using `build_force_vector` and the `f_chirp` force time history
-  Second project the force force onto the modal space using `project_force` 
-  Next calculate the response of the 3 SDOF systems to the modal excitation using `response_of_many_SDOF`
-  Plot all 3 Modal responses in the same plot (ax) and label each curve with the corresponding DOF number and its resonance frequency (`freq_modal`), 
`    ax.plot(t, modal_responses[i], label=f'Mode {i+1} - freq = {freq_modal[i]:.2f} Hz',alpha=0.5)`
</div>

In [ ]:
# recompute the modal frequencies
freq_modal = np.sqrt(np.diag(K_prj)/np.diag(M_prj))/(2*np.pi)

f_chirp_vector = None # you need to define the force vector here
f_chrip_prj = None # you need to define the projected force vector here
modal_responses = None # you need to define the modal responses here
# YOUR CODE HERE
raise NotImplementedError()

**Check your plot**

With chirp excitation, you excite your system with a signal that varies gradually in frequency. Passing through different frequencies as the signal continues. Around the second 95, our chirp reaches a  frequency of around 0.28 Hz and thus the resonance frequency of the SDOF mode 1, we see the amplitude of the mode 1 response reach a maximum at this time. As time progresses the amplitude of the mode 1 response decreases again. We see similar behaviour for Mode 2 and Mode 3 when the chirp passes to their resonance frequencies.

---

Let's recap what we have done so far: we projected the MDOF system onto the modal space, calculated the impulse responses of the 3 SDOF systems, and projected the force onto the modal space. 
Now that we have the response in the modal space, we can project it back to the physical space using the mode shapes.
<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.11:** 
Build a function `inverse_project_response` that takes as input the modal response and the mode shapes and returns the response in the original physical coordinates.
</div>



In [ ]:
def inverse_project_response(modal_responses:np.ndarray, psi:np.ndarray):
    """
    A function that calculates the physical response given the modal response and the modal shapes
    
    Arguments:
    modal_responses -- the modal response
    psi -- matrix of real modal shapes
    
    Returns:
    physical_responses -- the physical response
    """
    # YOUR CODE HERE
    raise NotImplementedError()

Lets test your implementation

In [ ]:
t_test = np.linspace(0, 600, 6000)
impulse_responses_test = np.array([[1,1,1,1,0,-1], 
                                    [1,1,1, 1,1,1],
                                    [1,0,1,0,1,1]])
f_modal_test = np.array([0,2, 3])
response_test = response_of_many_SDOF(impulse_responses_test, f_modal_test)
psi_2dof = np.array([[1,1],[1,-1]]) # Textbook mode shapes of a 2DOF system
modal_response_test = np.array([[-1,-1],[-1,1]]) # Modal response
assert np.allclose(response_test.mean(axis=-1), np.array([0. , 0.2, 0.2])), 'The response of the SDOF systems is not correct'
assert np.allclose(inverse_project_response(psi_2dof, modal_response_test),np.diag([-2,-2])) 

<div style="border-left: 1px solid black; padding: 1em; margin: 1em 0;">

**Assignment 1.14:** Project the response back to the physical space using `inverse_project_response` and plot the response of each DOF to the swept sine signal applied to the third DOF. Save the response of each DOF in a list called `system_responses` and save the the response of the third DOF in a variable called `system_response_3`.
</div>

In [ ]:
system_responses = None 
system_response_3 = None
fig,ax=plt.subplots(figsize=(10,5))

# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
assert system_responses is not None, 'system_responses is not defined'
assert system_response_3 is not None, 'system_response_3 is not defined'
assert len(system_response_3) == len(t), 'system_response_3 does not have the correct length'
assert np.isclose(np.square(system_response_3).mean(), 0.29539,rtol=1e-3), 'system_response_3 is not correct'
assert np.isclose(np.square(system_responses).mean(), 0.18206,rtol=1e-3), 'system_responses is not correct'

> ***EXTRA:***
We can actually compute the response of the system in the nodal space using the impulse response in the nodal space. This approach was not used in the course, but it is a nice exercise to see how the impulse response in the nodal space can be used to compute the response of the system.

In [ ]:
import scipy.signal as signal
f_s = 10 # Defining a sampling frequency
dt = 1/f_s # time step
t_extra = np.linspace(0,600, f_s*600) # Defining a time vector of 600s, sampled at f_s Hz
f_chirp_vector_extra = signal.chirp(t_extra, 0, max(t_extra), 2, phi=90)  # f_chirp_vector_extra = signal.chirp(t_extra, 0, max(t_extra), 2, phi=90)
system_response_3_extra = signal.convolve(h_nodal[:,2,2], f_chirp_vector_extra, mode='full')[:len(t_extra)]*dt
fig, ax = plt.subplots(2,figsize=(10,5))
ax[0].plot(t_extra, system_response_3_extra,alpha =0.5, label='System Response DOF 3 - Extra')
ax[1].plot(t, system_response_3, label='System Response DOF 3',alpha =0.5)
#ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('System Response\n (using the nodal response)')
ax[1].set_xlabel('Time (s)')
ax[1].set_ylabel('System Response\n (using the model superposition)')
ax[0].legend()
ax[1].legend()

### Frequency Response <a class="anchor" id="freq"></a>

Now let us have a look at the spectrum of this response using FFT, we clearly see how the 3DOFs contribute to the global response.


In [ ]:
f = np.fft.fftfreq(t.shape[-1],1/f_s)
F_system_response_3 = np.fft.fft(system_response_3)

fh = int(len(f)/2)

fig, ax1 = plt.subplots()

for i, x_i in enumerate(modal_responses):
    F_x_i = np.fft.fft(x_i)
    ax1.semilogy(f[:fh], np.abs(F_x_i[:fh]), label=f'$X_{i+1}(f)$', linewidth=1)

ax1.semilogy(f[:fh], np.abs(F_system_response_3[:fh]), label='$X(f)$')

plt.xlabel('Frequency (Hz)')
plt.ylabel('$X$(f)')
plt.xlim([0,2])
plt.grid(which='both', linestyle=':')
l = plt.legend()

In fact we also obtain a similar result in the frequency domain. For this we use the frequency response function. We simply need to compute the FFT of the original input signal and multiply it with the FRF between the input location and output location. 


In [ ]:
from session3 import frequency_response_function

f = np.fft.fftfreq(t.shape[-1],1/f_s)
F_T = np.fft.fft(f_chirp)# Transforming the Chirp signal to the frequency domain
X_from_mat = frequency_response_function(M,C,K,f=f, i=2,j=2)*F_T

plt.figure(figsize=(7,5))
plt.semilogy(f, abs(X_from_mat), label='$X(f)$ - Computed from H(f)F(f)')
plt.semilogy(f[:fh], np.abs(F_system_response_3[:fh]), label='$X(f)$', color='tab:red')
plt.xlabel('Frequency (Hz)')
plt.ylabel('$X$(f)')
plt.xlim([0,2])
plt.ylim(bottom=5e-4)
plt.grid(which='both', linestyle=':')
_ = plt.legend()

<div class="alert alert-block alert-success">
<b>Conclusion:</b> Through the application of modal projection, this session has demonstrated that it is possible to simulate the behavior of a multi-degree-of-freedom (MDOF) system by employing the superposition of single-degree-of-freedom (SDOF) systems.
</div>